In [ ]:
# Core libraries
from pathlib import Path
import sys
import math
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

In [ ]:
PROJECT_ROOT = Path.cwd().parent
HIST_DATA_PATH = PROJECT_ROOT / "data/raw/ncr_weather_historical.parquet"
EVAL_DATA_PATH = PROJECT_ROOT / "data/raw/ncr_weather_2026_present.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Historical path:", HIST_DATA_PATH)
print("Evaluation path:", EVAL_DATA_PATH)

In [ ]:
historical = pd.read_parquet(HIST_DATA_PATH)
print("Historical data shape:", historical.shape)
print("Historical data columns:", historical.columns.tolist())
historical.head()

In [ ]:
FEATURES = ["temperature", "pressure", "humidity"]
station = historical["station_id"].value_counts().index[0]
station_df = historical[historical["station_id"] == station].copy()

print("Station:", station)
print("Rows:", len(station_df))
print("Start:", station_df["timestamp"].min())
print("End:", station_df["timestamp"].max())
station_df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col in zip(axes, FEATURES):
    ax.plot(station_df["timestamp"], station_df[col])
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Timestamp")
fig.suptitle(f"Weather Data for Station {station}", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
TRAIN_SPLIT = 0.8
split_idx = int(len(station_df) * TRAIN_SPLIT)
train_df = station_df.iloc[:split_idx]
val_df = station_df.iloc[split_idx:]
X_train_raw = train_df[FEATURES].to_numpy(dtype=np.float32)
X_val_raw = val_df[FEATURES].to_numpy(dtype=np.float32)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val_scaled = scaler.transform(X_val_raw).astype(np.float32)


print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)

In [ ]:
WINDOW_SIZE = 24

def create_windows(data, window_size):
    windows = []
    for i in range(len(data)-window_size+1):
        windows.append(data[i:i+window_size])
    if not windows:
        return np.empty((0, window_size, data.shape[1]), dtype=data.dtype)
    return np.stack(windows)

train_windows = create_windows(X_train_scaled, WINDOW_SIZE)
val_windows = create_windows(X_val_scaled, WINDOW_SIZE)

print("Train windows shape:", train_windows.shape)
print("Validation windows shape:", val_windows.shape)
print("First train window:\n", train_windows[0])

In [ ]:
train_dataset = TensorDataset(torch.from_numpy(train_windows))
val_dataset = TensorDataset(torch.from_numpy(val_windows))

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))[0]
print("Batch shape:", batch[0].shape)

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=3, hidden_size=32, num_layers=1):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, c_n) = self.encoder(x)
        latent = h_n[-1]
        decoder_input = latent.unsqueeze(1).repeat(1, x.size(1), 1)
        decoded, _ = self.decoder(decoder_input)
        reconstructed = self.output_layer(decoded)
        return reconstructed

    @torch.no_grad()
    def encode(self, x):
        _, (h_n, c_n) = self.encoder(x)
        return h_n[-1]

model = LSTMAutoencoder(n_features=len(FEATURES), hidden_size=32, num_layers=1).to(DEVICE)
print(model)
print("Number of parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
model.eval()
with torch.no_grad():
    x0 = batch[:4].to(DEVICE)
    x_hat0 = model(x0)
    z0 = model.encode(x0)

print("Input shape:", x0.shape)
print("Latent shape:", z0.shape)
print("Reconstructed shape:", x_hat0.shape)
print("MAE reconstruction error:", torch.mean(torch.abs(x0 - x_hat0)).item())


In [ ]:
EPOCHS = 20
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

history = {"train_loss": [], "val_loss": []}
best_state = None
best_val = math.inf

for epoch in range(1, EPOCHS+1):
    model.train()
    train_losses = []
    for (x,) in train_loader:
        x = x.to(DEVICE)
        optimizer.zero_grad()
        x_hat = model(x)
        loss = criterion(x_hat, x)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    model.eval()
    val_losses = []
    with torch.no_grad():
        for (x,) in val_loader:
            x = x.to(DEVICE)
            x_hat = model(x)
            loss = criterion(x_hat, x)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if(val_loss < best_val):
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())

    if epoch==1 or epoch%5 ==0 or epoch==EPOCHS:
        print(f"Epoch {epoch}/{EPOCHS} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.title("LSTM Autoencoder Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
@torch.no_grad()
def reconstruct_windows(model, windows, batch_size=256):
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=batch_size, shuffle=False)
    outputs = []
    model.eval()
    for (x,) in loader:
        x = x.to(DEVICE)
        x_hat = model(x)
        outputs.append(x_hat.cpu().numpy())
    return np.concatenate(outputs, axis=0)

val_recon = reconstruct_windows(model, val_windows)

sample_id = min(3, len(val_windows)-1)
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for i, feature in enumerate(FEATURES):
    axes[i].plot(val_windows[sample_id, :, i], label="Original", color="blue")
    axes[i].plot(val_recon[sample_id, :, i], label="Reconstructed", color="orange")
    axes[i].set_ylabel(feature)
    axes[i].grid(alpha=0.2)
axes[-1].set_xlabel("Time Step")
fig.suptitle(f"Original vs Reconstructed for Sample {sample_id}", fontsize=16)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
window_scores = np.mean(np.abs(val_windows - val_recon)**2, axis=(1, 2))
threshold = np.percentile(window_scores, 99)

print("Validation scores summary:")
print(pd.Series(window_scores).describe())
print(f"Threshold for anomaly detection (99th percentile):{threshold:.6f}")
print(f"Number of anomalies detected: {np.sum(window_scores > threshold)} out of {len(window_scores)} windows")

plt.figure(figsize=(9, 4))
plt.hist(window_scores, bins=50, color="skyblue", edgecolor="black")
plt.axvline(threshold, color="red", linestyle="--", label=f"99th Percentile")
plt.title("Validation reconstruction error distribution")
plt.xlabel("Reconstruction Error (MSE)")
plt.ylabel("Count")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# Test with simple anomalies
def inject_demo_anomalies(df, seed=SEED):
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out)
    labels = np.zeros(n, dtype=bool)
    types = np.array(["normal"] * n, dtype=object)

    spike_i = n // 3
    offset_start = n // 2
    offset_end = min(offset_start + 6, n)
    drift_start = (2 * n) // 3
    drift_end = min(drift_start + 12, n)

    out.loc[spike_i, "temperature"] += 10.0
    labels[spike_i] = True
    types[spike_i] = "spike"

    out.loc[offset_start:offset_end - 1, "humidity"] += 15.0
    labels[offset_start:offset_end] = True
    types[offset_start:offset_end] = "offset"

    ramp = np.linspace(0, 8.0, drift_end - drift_start)
    out.loc[drift_start:drift_end - 1, "pressure"] += ramp
    labels[drift_start:drift_end] = True
    types[drift_start:drift_end] = "drift"

    return out, labels, types

val_corrupt_df, point_labels, point_types = inject_demo_anomalies(val_df.reset_index(drop=True))

val_corrupt_scaled = scaler.transform(val_corrupt_df[FEATURES]).astype(np.float32)
corrupt_windows = create_windows(val_corrupt_scaled, WINDOW_SIZE)
corrupt_recon = reconstruct_windows(model, corrupt_windows)
corrupt_scores = np.mean(np.abs(corrupt_windows - corrupt_recon)**2, axis=(1, 2))

windows_pred = corrupt_scores > threshold
print(f"Anomalous timestamps injected: {point_labels.sum()} out of {len(point_labels)}")
print(f"Flagged Windows: {windows_pred.sum()} out of {len(windows_pred)}")

In [ ]:
# Visualize one channel and the synthetic anomaly locations.
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, feature in zip(axes, FEATURES):
    ax.plot(val_df.reset_index(drop=True)[feature], label="clean", linewidth=0.8)
    ax.plot(val_corrupt_df[feature], label="corrupted", linewidth=0.8, alpha=0.8)
    ax.set_ylabel(feature)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("validation timestamp index")
fig.suptitle("Controlled anomalies used for the experiment")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(window_scores, bins=40, alpha=0.8, label="clean validation")
axes[0].hist(corrupt_scores, bins=40, alpha=0.6, label="corrupted validation")
axes[0].axvline(threshold, linestyle="--", label="threshold")
axes[0].set_xlabel("window reconstruction MSE")
axes[0].set_ylabel("count")
axes[0].set_title("Score distributions")
axes[0].legend()

axes[1].plot(corrupt_scores, linewidth=0.9)
axes[1].axhline(threshold, linestyle="--", label="threshold")
axes[1].set_xlabel("window index")
axes[1].set_ylabel("reconstruction MSE")
axes[1].set_title("Corrupted-series scores")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
def window_labels_from_point_labels(point_labels, window_size):
    labels = []
    for start in range(len(point_labels) - window_size + 1):
        labels.append(bool(point_labels[start:start + window_size].any()))
    return np.array(labels, dtype=bool)

window_true = window_labels_from_point_labels(point_labels, WINDOW_SIZE)
window_pred = corrupt_scores > threshold

tn, fp, fn, tp = confusion_matrix(window_true, window_pred, labels=[False, True]).ravel()
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"Precision={precision:.3f}")
print(f"Recall={recall:.3f}")
print(f"F1={f1:.3f}")

In [ ]:
CANDIDATE_WINDOWS = [12, 24, 48]

comparison = []
for w in CANDIDATE_WINDOWS:
    if len(val_windows) == 0 or w > len(X_val_scaled):
        continue
    tw = create_windows(X_train_scaled, w)
    vw = create_windows(X_val_scaled, w)
    print(f"window={w:>2} | train shape={tw.shape} | validation shape={vw.shape}")
    comparison.append((w, len(tw), len(vw)))

pd.DataFrame(comparison, columns=["window_size", "n_train_windows", "n_val_windows"])

In [ ]:
capacity_rows = []
for hidden in [8, 16, 32, 64]:
    trial = LSTMAutoencoder(n_features=3, hidden_size=hidden).to(DEVICE)
    with torch.no_grad():
        trial_out = trial(batch.to(DEVICE))
        initial_loss = criterion(trial_out, batch.to(DEVICE)).item()
    n_params = sum(p.numel() for p in trial.parameters())
    capacity_rows.append({"hidden_size": hidden, "parameters": n_params, "untrained_batch_mse": initial_loss})

pd.DataFrame(capacity_rows)

In [ ]:
if PROJECT_ROOT is not None:
    try:
        sys.path.insert(0, str(PROJECT_ROOT / "src"))
        from skyguard.simulation.anomaly_injector import AnomalyConfig, inject_anomalies
        print("Imported SkyGuard anomaly injector successfully.")
        print("Available config class:", AnomalyConfig)
    except Exception as exc:
        print("Optional import skipped:", repr(exc))
else:
    print("Not running inside a SkyGuard repository; skip this optional integration cell.")

In [ ]:
experiment_df = (
    val_df
    .sort_values("timestamp")
    .iloc[:24 * 60]       # roughly 60 days of hourly data
    .copy()
)

corrupted_df, injection_log = inject_anomalies(
    experiment_df,
    variables=["temperature", "pressure", "humidity"],
    config=AnomalyConfig(),
)

print(f"Injected {len(injection_log)} anomaly events")
display(injection_log)

In [ ]:
corrupted_scaled = scaler.transform(corrupted_df[FEATURES]).astype(np.float32)
corrupted_windows = create_windows(corrupted_scaled, WINDOW_SIZE)
corrupted_recon = reconstruct_windows(model, corrupted_windows)
corrupted_scores = np.mean(np.abs(corrupted_windows - corrupted_recon)**2, axis=(1, 2))

print("Corrupted scores summary:")
print(pd.Series(corrupted_scores).describe())
corrupted_preds = corrupted_scores > threshold
print(f"Threshold for anomaly detection (99th percentile):{threshold:.6f}")
print("Flagged windows:", np.sum(corrupted_preds), "out of", len(corrupted_preds))
print(f"{corrupted_preds.mean():.2%}")


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(corrupted_scores, linewidth=0.8, label="Corrupted window reconstruction error")
plt.axhline(
    threshold, linestyle="--", label=f"Clean validation P99 threshold ({threshold:.4f})"
)

flagged = np.where(corrupted_preds)[0]
plt.scatter(flagged, corrupted_scores[flagged], s=12, label="Flagged windows")
plt.xlabel("Window index")
plt.ylabel("Reconstruction MSE")
plt.title("LSTM Autoencoder Scores on SkyGuard-Injected Anomalies")
plt.grid(alpha=0.2)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.hist(window_scores, bins=60, alpha=0.8, label="clean validation")
plt.hist(corrupted_scores, bins=60, alpha=0.6, label="corrupted validation")
plt.axvline(threshold, color="red", linestyle="--", label=f"Clean validation P99 threshold ({threshold:.4f})")
plt.xlabel("Window reconstruction MSE")
plt.ylabel("Density")
plt.title("Score Distributions: Clean vs Corrupted Validation")
plt.legend()
plt.grid(alpha=0.2)
plt.show()